In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121

[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python3 -m pip install --upgrade pip


In [1]:
import torch
from tokenizer import Tokenizer
from model import Model

In [2]:
my_tokenizer = Tokenizer("tokenizer.json")
prompt = "Merhaba, senin adın ne söyleyebilir misin?"
tokens = my_tokenizer.encode(prompt)
torch.manual_seed(2)
model = Model(len(my_tokenizer.vocab), embedding_dim=4)
model(tokens)  # batch dimension ekle

tensor([[[-0.0708,  0.0557,  0.4198, -0.1013],
         [-0.2616,  0.2770,  0.1422,  0.0398],
         [ 0.0343,  0.2076,  0.6253,  0.1278],
         [-0.0071, -0.0583,  0.5213, -0.1854],
         [-0.1629,  0.2731,  0.2537,  0.0304],
         [-0.1469,  0.2801,  0.3116,  0.0776],
         [ 0.0361,  0.1685,  0.6358,  0.1065],
         [-0.0311,  0.0397,  0.4912, -0.1047],
         [-0.1464,  0.1702,  0.2636, -0.0843],
         [-0.1167,  0.2578,  0.3645,  0.0661],
         [-0.0334,  0.1509,  0.4996,  0.0284],
         [-0.1566,  0.3401,  0.2941,  0.1314],
         [-0.0502, -0.0311,  0.4808, -0.1421],
         [-0.3810,  0.4325, -0.0963,  0.0629],
         [-0.1422,  0.1723,  0.2658, -0.0838],
         [-0.1913,  0.4081,  0.2479,  0.1795],
         [ 0.0618,  0.0191,  0.6495, -0.0571],
         [-0.2893,  0.3949,  0.1121,  0.1498],
         [-0.3631,  0.2222, -0.1009, -0.1649],
         [-0.2710,  0.2006,  0.1020, -0.0806],
         [-0.1979,  0.3613,  0.2287,  0.1360],
         [-0.

In [8]:
sentence_meaning = model(tokens)



In [6]:
all_similarities= torch.zeros((sentence_meaning.shape[0],sentence_meaning.shape[0]))

for i in range(sentence_meaning.shape[0]):
    similarities = []
    for j in range(sentence_meaning.shape[0]):
        sim = sentence_meaning[i][0]*sentence_meaning[j][0]+sentence_meaning[i][1]*sentence_meaning[j][1]+sentence_meaning[i][2]*sentence_meaning[j][2]+sentence_meaning[i][3]*sentence_meaning[j][3]
        similarities.append(sim)
    all_similarities[i] = torch.tensor(similarities)
all_similarities.detach().numpy()

NameError: name 'sentence_meaning' is not defined

In [5]:
sentence_meaning.shape[1]

NameError: name 'sentence_meaning' is not defined

In [15]:
all_similarities= torch.zeros((sentence_meaning.shape[0],sentence_meaning.shape[0]))
sentence_meaning=sentence_meaning.squeeze(0)
for i in range(sentence_meaning.shape[0]):
    i_similarities = torch.zeros(sentence_meaning.shape[0])
    for j in range(sentence_meaning.shape[0]):
        for k in range(sentence_meaning.shape[1]):
            i_similarities[j] += sentence_meaning[i][k]*sentence_meaning[j][k]
    all_similarities[i] = i_similarities
all_similarities.detach().numpy()

RuntimeError: The expanded size of the tensor (1) must match the existing size (24) at non-singleton dimension 0.  Target sizes: [1].  Tensor sizes: [24]

In [11]:
sentence_meaning.shape

torch.Size([1, 24, 4])

In [10]:
# query, key, value yaklaşımıyla benzerlik hesaplama
sentence_meaning=sentence_meaning.squeeze(0)

all_similarities=sentence_meaning @ sentence_meaning.T # her bir kelime diğer kelimelerle olan benzerlikleri gösterir



In [11]:
# yukarıda her bir kelime diğer kelimelere ne kadar benziyoru gördük şimdi de her bir kelime bu cümle içinde ne kadar önemli

attention_weights = torch.softmax(all_similarities, dim=1)
attention_weights

tensor([[0.0414, 0.0418, 0.0414, 0.0417, 0.0417, 0.0417, 0.0418, 0.0419, 0.0417,
         0.0417, 0.0418, 0.0418, 0.0419, 0.0415, 0.0414, 0.0417, 0.0413, 0.0419,
         0.0416, 0.0418, 0.0418, 0.0417, 0.0416, 0.0414],
        [0.0411, 0.0419, 0.0410, 0.0418, 0.0419, 0.0417, 0.0420, 0.0422, 0.0416,
         0.0416, 0.0421, 0.0421, 0.0421, 0.0413, 0.0412, 0.0416, 0.0409, 0.0422,
         0.0416, 0.0418, 0.0420, 0.0417, 0.0416, 0.0411],
        [0.0400, 0.0404, 0.0425, 0.0418, 0.0419, 0.0429, 0.0419, 0.0419, 0.0412,
         0.0413, 0.0425, 0.0409, 0.0419, 0.0420, 0.0404, 0.0405, 0.0407, 0.0440,
         0.0435, 0.0445, 0.0414, 0.0403, 0.0407, 0.0409],
        [0.0403, 0.0410, 0.0417, 0.0419, 0.0419, 0.0424, 0.0420, 0.0422, 0.0414,
         0.0414, 0.0424, 0.0415, 0.0421, 0.0416, 0.0405, 0.0410, 0.0406, 0.0434,
         0.0427, 0.0435, 0.0418, 0.0409, 0.0410, 0.0409],
        [0.0402, 0.0411, 0.0417, 0.0418, 0.0421, 0.0425, 0.0419, 0.0420, 0.0414,
         0.0412, 0.0426, 0.0416, 0.0420

In [12]:
sentence_context_vectors = attention_weights @ sentence_meaning
sentence_context_vectors

tensor([[ 0.2023, -0.0484, -0.0582,  0.0136],
        [ 0.2025, -0.0485, -0.0582,  0.0136],
        [ 0.2043, -0.0470, -0.0598,  0.0135],
        [ 0.2037, -0.0476, -0.0591,  0.0135],
        [ 0.2037, -0.0476, -0.0592,  0.0137],
        [ 0.2046, -0.0470, -0.0599,  0.0136],
        [ 0.2037, -0.0477, -0.0590,  0.0135],
        [ 0.2037, -0.0478, -0.0590,  0.0135],
        [ 0.2032, -0.0479, -0.0588,  0.0136],
        [ 0.2033, -0.0479, -0.0588,  0.0135],
        [ 0.2043, -0.0473, -0.0596,  0.0136],
        [ 0.2030, -0.0482, -0.0585,  0.0136],
        [ 0.2037, -0.0478, -0.0590,  0.0135],
        [ 0.2039, -0.0473, -0.0594,  0.0136],
        [ 0.2026, -0.0482, -0.0585,  0.0137],
        [ 0.2026, -0.0484, -0.0583,  0.0135],
        [ 0.2029, -0.0479, -0.0587,  0.0136],
        [ 0.2054, -0.0465, -0.0605,  0.0136],
        [ 0.2050, -0.0466, -0.0603,  0.0136],
        [ 0.2058, -0.0462, -0.0607,  0.0135],
        [ 0.2033, -0.0480, -0.0588,  0.0135],
        [ 0.2025, -0.0485, -0.0582

In [ ]:
"""u_tokeneizer= Tokenizer("tokenizer.json")
tokens = u_tokeneizer.encode("Merhaba, senin adın ne söyleyebilir misin?")
torch.manual_seed(2)
model= Model(len(u_tokeneizer.vocab), embedding_dim=4,device='cpu')
sentence_meaning= model(tokens)

attention_weights = torch.softmax(sentence_meaning @ sentence_meaning.T, dim=1)
sentence_context_vectors = attention_weights @ sentence_meaning"""

'u_tokeneizer= Tokenizer("tokenizer.json")\ntokens = u_tokeneizer.encode("Merhaba, senin adın ne söyleyebilir misin?")\ntorch.manual_seed(2)\nmodel= Model(len(u_tokeneizer.vocab), embedding_dim=4,device=\'cpu\')\nsentence_meaning= model(tokens)\n\nattention_weights = torch.softmax(sentence_meaning @ sentence_meaning.T, dim=1)\nsentence_context_vectors = attention_weights @ sentence_meaning'

In [13]:
q_weights = torch.nn.Linear(4, 3,bias=False) # 4 boyutlu embeddingi 3 boyuta indiriyoruz
k_weights = torch.nn.Linear(4, 3,bias=False)
v_weights = torch.nn.Linear(4, 3,bias=False)


q_of_sentence = q_weights(sentence_meaning)
k_of_sentence = k_weights(sentence_meaning)
v_of_sentence = v_weights(sentence_meaning)

q_of_sentence.shape, k_of_sentence.shape, v_of_sentence.shape

(torch.Size([24, 3]), torch.Size([24, 3]), torch.Size([24, 3]))

In [ ]:
"""from plot_tokens import plot_tokens

u_sentences =[
  {  "words":q_of_sentence.detach().numpy(),
    "labels":my_tokenizer.tokenize(prompt),
    "color":"blue",},
    {
    "words":k_of_sentence.detach().numpy(),
    "labels":my_tokenizer.tokenize(prompt),
    "color":"red",
    },
    {
    "words":v_of_sentence.detach().numpy(),
    "labels":my_tokenizer.tokenize(prompt),
    "color":"green",
    }
]
plot_tokens(u_sentences, title="Query, Key, Value")"""

'from plot_tokens import plot_tokens\n\nu_sentences =[\n  {  "words":q_of_sentence.detach().numpy(),\n    "labels":my_tokenizer.tokenize(prompt),\n    "color":"blue",},\n    {\n    "words":k_of_sentence.detach().numpy(),\n    "labels":my_tokenizer.tokenize(prompt),\n    "color":"red",\n    },\n    {\n    "words":v_of_sentence.detach().numpy(),\n    "labels":my_tokenizer.tokenize(prompt),\n    "color":"green",\n    }\n]\nplot_tokens(u_sentences, title="Query, Key, Value")'

In [25]:
#şimdi query ile key arasında benzerlik hesaplayalım

k_of_sentence.shape[-1] # son boyut aynı olmalı

3

In [14]:
attention_scores = q_of_sentence @ k_of_sentence.T
attention_weights = torch.softmax(attention_scores / torch.sqrt(torch.tensor(k_of_sentence.shape[-1], dtype=torch.float32)), dim=1)

In [15]:
context_vector= attention_weights @ v_of_sentence
context_vector  # artık burda bir bağlam ve bilgi var.
# sözlük bilgisini embedding tarafında asıl bağlam da burda tutulur

tensor([[ 0.0343, -0.0525,  0.0598],
        [ 0.0343, -0.0525,  0.0598],
        [ 0.0343, -0.0524,  0.0598],
        [ 0.0343, -0.0524,  0.0598],
        [ 0.0343, -0.0524,  0.0598],
        [ 0.0343, -0.0524,  0.0598],
        [ 0.0343, -0.0525,  0.0598],
        [ 0.0343, -0.0525,  0.0598],
        [ 0.0343, -0.0525,  0.0598],
        [ 0.0343, -0.0525,  0.0598],
        [ 0.0343, -0.0524,  0.0598],
        [ 0.0343, -0.0525,  0.0598],
        [ 0.0343, -0.0524,  0.0598],
        [ 0.0343, -0.0524,  0.0598],
        [ 0.0343, -0.0525,  0.0598],
        [ 0.0343, -0.0525,  0.0598],
        [ 0.0343, -0.0525,  0.0598],
        [ 0.0343, -0.0524,  0.0598],
        [ 0.0343, -0.0524,  0.0598],
        [ 0.0343, -0.0524,  0.0598],
        [ 0.0343, -0.0525,  0.0598],
        [ 0.0343, -0.0525,  0.0598],
        [ 0.0343, -0.0525,  0.0598],
        [ 0.0343, -0.0525,  0.0598]], grad_fn=<MmBackward0>)

In [17]:
# causal self attention
# sadece geçmişteki kelimelere bakarak benzerlik hesaplama

mask= torch.tril(torch.ones((attention_weights.shape[0],attention_weights.shape[1])))  # alt üçgen maske oluşturma
#torch.softmax(mask*attention_weights,dim=1) #burda o yapınca softmaxte bir değere karşılık geliyor. Ama biz maskeleme yapmak istediğimizden dolayı spoftmaxten çıktısınında 0 olmasını istediğimden 0'ların hepsini - sonsuz yapıyorum

masked_attention_weights= attention_weights.masked_fill(mask==0, float('-inf'))
softmaxe_attention_weights= torch.softmax(masked_attention_weights, dim=1)
# bu aşamadan sonra bazı durumlarda baskın olabilecek hücreler olabiliyor. Bunlar olmasın diye dropout işlemiyle rastgele kapatabiliyoruz. Bu sayede eğitim aşamasında herhangi bir şeye odaklanmasının önüne geçiyoruz rastgele tahmin yapmasını sağlıyoruz.


In [18]:
dropout_rate= 0.5 # parametrelerin yüzde 50sini dropout yap
torch.manual_seed(42)
dropout= torch.nn.Dropout(dropout_rate)
dropout(softmaxe_attention_weights)

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.6667, 0.6666, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4000, 0.0000, 0.4000, 0.4000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000

In [ ]:
from türkçe_dil_modelim.multi_head_attention_old import MultiHeadAttention
import torch
import torch.nn as nn
    
multi_head_attention = MultiHeadAttention(embed_size=4, output_size=8, num_heads=2, context_length=10, dropout_rate=0.5)
multi_head_attention(torch.randn(4,4)).shape


torch.Size([4, 8])